## Last Try to increase validation acc — Architecture Change to ConvNeXt-Small

New architecture with greater spatial abstraction capacity. **ConvNeXt-Small** :

- **Large Kernels (`7 × 7`):** Unlike EfficientNet (`3 × 3`), ConvNeXt analyzes much larger areas of the image in a single step, allowing it to better capture the global structure of paragraphs and the continuity of letters.
- **Transformer-Style Mechanisms:** It follows Vision Transformers (such as Inverted Bottleneck layers and GELU activations) while maintaining the efficiency of convolutional networks.
- **Superior Texture Representation:** better performance than EfficientNet on classification benchmarks involving documents and art history.

In [ ]:
import os, glob, time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image, ImageFile

Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Dispositivo activo: {device}")

BATCH_SIZE = 16          
IMG_SIZE = 384           
EPOCHS = 20              
LEARNING_RATE = 5e-5     # LR más bajo para ConvNeXt


# 2. Metadata y Dataset

DATA_DIR = "/kaggle/input/datasets/mariajgamboa/clammdst/CLaMM_DST"
CSV_PATH = os.path.join(DATA_DIR, "CLaMM_Metadata.csv")
df = pd.read_csv(CSV_PATH, sep=None, engine='python')
df.columns = df.columns.str.strip()
script_col = [col for col in df.columns if col.upper() == 'SCRIPT_TYPE'][0]
df['label'] = df[script_col].astype(int) - 1
num_classes = df['label'].nunique()

all_image_paths = (
    glob.glob(f"{DATA_DIR}/**/*.[jJ][pP][gG]", recursive=True) +
    glob.glob(f"{DATA_DIR}/**/*.[pP][nN][gG]", recursive=True) +
    glob.glob(f"{DATA_DIR}/**/*.[tT][iI][fF]", recursive=True) +
    glob.glob(f"{DATA_DIR}/**/*.[jJ][pP][eE][gG]", recursive=True))
image_map = {os.path.basename(p).lower(): p for p in all_image_paths}

class CLaMMDataset(Dataset):
    def __init__(self, df, image_map, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_map = image_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['FILENAME']).strip().lower()

        if img_name in self.image_map:
            final_path = self.image_map[img_name]
        else:
            base_name = os.path.splitext(img_name)[0]
            matched = None
            for ext in ['.jpg', '.jpeg', '.tif', '.png']:
                if (base_name + ext) in self.image_map:
                    matched = self.image_map[base_name + ext]
                    break
            final_path = matched

        image = Image.open(final_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(row['label'], dtype=torch.long)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = CLaMMDataset(df=df, image_map=image_map, transform=train_tf)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset_split = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42))

val_dataset = CLaMMDataset(df=df, image_map=image_map, transform=val_tf)
val_dataset = torch.utils.data.Subset(val_dataset, val_dataset_split.indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# Pérdida Ponderada

class_counts = df['label'].value_counts().sort_index().values
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * num_classes
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)


# ConvNeXt-Small
print("\n Cargando ConvNeXt-Small preentrenado...")
model = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)

# Reemplazar la capa clasificadora final de ConvNeXt
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


# training loop

best_val_acc = 0.0
save_filename = "convnext_small_384_clamm.pth"

print(f"\n Iniciando FASE 4 ({EPOCHS} épocas con ConvNeXt-Small)...\n")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data)
        total_train += labels.size(0)

    scheduler.step()

    train_loss = running_loss / total_train
    train_acc = (correct_train.double() / total_train).item()

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data)
            total_val += labels.size(0)

    val_loss = val_loss / total_val
    val_acc = (correct_val.double() / total_val).item()

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Época [{epoch+1:02d}/{EPOCHS:02d}] | LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_filename)
        print(f"   └─ ¡Nuevo récord ConvNeXt-Small: {best_val_acc * 100:.2f}%! Modelo guardado.")

elapsed = time.time() - start_time
print(f"\n FASE 4 Completada en {elapsed/60:.2f} minutos.")
print(f" Mejor Accuracy Final (ConvNeXt-Small): {best_val_acc * 100:.2f}%")

 Dispositivo activo: cuda

 Cargando ConvNeXt-Small preentrenado...
Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:01<00:00, 187MB/s]



 Iniciando FASE 4 (20 épocas con ConvNeXt-Small)...

Época [01/20] | LR: 0.000050 | Train Loss: 1.7435 | Train Acc: 38.74% | Val Loss: 1.5355 | Val Acc: 45.59%
   └─ ¡Nuevo récord ConvNeXt-Small: 45.59%! Modelo guardado.
Época [02/20] | LR: 0.000049 | Train Loss: 1.2172 | Train Acc: 58.76% | Val Loss: 1.1155 | Val Acc: 65.75%
   └─ ¡Nuevo récord ConvNeXt-Small: 65.75%! Modelo guardado.
Época [03/20] | LR: 0.000047 | Train Loss: 1.0642 | Train Acc: 65.39% | Val Loss: 1.0648 | Val Acc: 63.22%
Época [04/20] | LR: 0.000045 | Train Loss: 0.9675 | Train Acc: 70.18% | Val Loss: 1.0156 | Val Acc: 72.36%
   └─ ¡Nuevo récord ConvNeXt-Small: 72.36%! Modelo guardado.
Época [05/20] | LR: 0.000043 | Train Loss: 0.8741 | Train Acc: 75.22% | Val Loss: 0.9920 | Val Acc: 72.58%
   └─ ¡Nuevo récord ConvNeXt-Small: 72.58%! Modelo guardado.
Época [06/20] | LR: 0.000040 | Train Loss: 0.8204 | Train Acc: 77.31% | Val Loss: 0.9719 | Val Acc: 72.58%
Época [07/20] | LR: 0.000037 | Train Loss: 0.7745 | Train Ac